In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import pickle
from sklearn.ensemble import RandomForestRegressor
from utils.data_utils import evaluate_rf
import utils.cross_validation as cval

from shapely.geometry import box

In [8]:
TEST_SIZE = 0.2
RANDOM_KEY = 42
BATCH_SIZE=16


df= pd.read_csv("data/final/fd_df.csv")
df.drop_duplicates(subset="PID", inplace=True)
PID_loc= pd.read_csv("data/lookup/PID_location_all.csv")


ecoregions=cval.process_ecoregion("data/Ecoregions/Ecoregions2017.shp")

df=df.merge(PID_loc, on="PID", how="left")
# df.dropna(subset=["lat", "lon"], inplace=True)

df = df.drop(columns=[col for col in df.columns if "_y" in col])
df.columns = df.columns.str.replace('_x$', '', regex=True)

df=cval.assign_spatial_groups(df, grid_size=1.0)

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"  # WGS84
)



In [6]:
df.drop_duplicates(subset="PID", inplace=True)

In [7]:
df

,PID,Raos_Q,Functional_Evenness,Mean Pairwise D,CHELSA_BIO_Annual_Mean_Temperature,CHELSA_BIO_Annual_Precipitation,CHELSA_BIO_Precipitation_Seasonality,CrowtherLab_SoilMoisture_intraAnnualSD_downsampled10km,SG_SOC_Content_015cm,EsaCci_BurntAreasProbability,...,WSCI,lat,lon,BHAGE,managed,ownership,biome,lon_bin,lat_bin,spatial_group
0,1_32_13_80818_1,3.228699,1.579574,6.033445,39.433038,404.598144,39.003315,21.217440,32.522411,0.000000,...,8.441968,41.673533,-118.744119,125.0,0.0,blm,Xeric shrublands,-119.0,41.0,Xeric shrublands_-119.0_41.0
42,1_32_17_80834_1,1.026364,1.961698,2.555784,104.532986,305.508911,43.399940,16.491831,12.469229,0.593955,...,8.208374,37.343281,-114.745049,45.0,0.0,blm,Xeric shrublands,-115.0,37.0,Xeric shrublands_-115.0_37.0
48,1_32_17_80951_1,0.966063,1.985641,2.555784,106.483831,381.716339,38.391670,19.912308,12.886782,0.000000,...,8.105085,37.490703,-114.482374,100.0,0.0,blm,Xeric shrublands,-115.0,37.0,Xeric shrublands_-115.0_37.0
54,1_32_17_89667_1,1.342537,1.304197,2.217293,52.702132,390.322937,21.995016,18.865871,30.780912,0.000000,...,8.450443,38.515443,-114.937861,300.0,0.0,blm,Temperate conifer forests,-115.0,38.0,Temperate conifer forests_-115.0_38.0
84,1_32_23_87150_1,0.849211,1.583434,2.045551,84.434293,313.037781,20.899969,13.235727,25.377388,0.000000,...,8.510671,38.212990,-115.633713,66.0,0.0,national_forest,Xeric shrublands,-116.0,38.0,Xeric shrublands_-116.0_38.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1247853,5_30_67_95852_1,1.795766,1.343936,2.333429,2.533751,631.706360,52.609997,25.848551,35.879810,0.000000,...,10.017184,46.012953,-110.405736,105.0,1.0,national_forest,Temperate conifer forests,-111.0,46.0,Temperate conifer forests_-111.0_46.0
1247895,5_30_93_91391_1,0.909966,0.767967,2.899007,-4.668354,601.542358,51.900021,21.084578,30.055454,0.000000,...,9.704644,45.815136,-112.816271,130.0,0.0,national_forest,Temperate conifer forests,-113.0,45.0,Temperate conifer forests_-113.0_45.0
1248005,5_30_93_92535_1,1.119445,1.528525,2.252641,13.216315,575.140625,46.503338,19.795029,30.715349,0.000000,...,9.923212,45.879784,-112.901646,153.0,0.0,national_forest,Temperate conifer forests,-113.0,45.0,Temperate conifer forests_-113.0_45.0
1248467,5_30_93_94959_1,0.536104,1.067395,2.333429,1.783094,611.599060,50.496704,20.000582,23.921318,0.000000,...,9.825863,45.863534,-112.808252,79.0,0.0,national_forest,Temperate conifer forests,-113.0,45.0,Temperate conifer forests_-113.0_45.0


In [ ]:
train, test= cval.ecoregion_cross_validation(gdf, ecoregions, TEST_SIZE, BATCH_SIZE)

train = train.loc[:, :'WSCI']
test = test.loc[:, :'WSCI']

y=['transformed npp', 'WSCI'] 


In [ ]:
X_train=train.drop(columns=y+['PID']).values
y_train = np.column_stack([train['transformed npp'], train['WSCI']])

X_test=test.drop(columns=y+['PID']).values
y_test = np.column_stack([test['transformed npp'], test['WSCI']])


fd_reg = RandomForestRegressor(random_state=RANDOM_KEY, n_jobs=4)
fd_reg.fit(X_train, y_train)

# sd_reg = RandomForestRegressor(random_state=RANDOM_KEY, n_jobs=4)
# sd_reg.fit(sX_train, sy_train)

evaluate_rf(X_test, y_test, fd_reg, feature_names=train.columns, importance= True, div_type= 'f')
# evaluate_rf(sX_test, sy_test, sd_reg, feature_names=sd_x.columns, importance= True,  div_type= 's')